In [2]:

"""
But : ce notebook implémente un moteur à base de règles (MBR) pour segmenter un texte en chunks.
Usage : exécuter les cellules dans l'ordre afin de lancer successivement les expérimentations sur les trois textes.
Remarque : la première expérimentation sert de base à la construction du système de règles,
la deuxième réutilise la même langue et la même source,
et la troisième transpose le lexique dans une autre langue en conservant les mêmes règles.
"""

# Modules utilisés pour la tokenisation, la lecture des ressources et la génération des sorties.
import re
from pandas import read_excel
import xml.etree.ElementTree as ET

def tokeniser(texte):
    """
    Découper un texte en tokens exploitables par le moteur de chunking.

    Entrées :
        texte (str) : texte brut à segmenter.
    Sorties :
        valeur de retour (list[str]) : liste ordonnée des tokens du texte.
    """
    # Uniformiser les apostrophes typographiques pour simplifier les règles de tokenisation.
    texte = texte.replace("\u2019", "'").replace("\u2018", "'")
    # La regex préserve notamment certaines élisions françaises, les mots composés et la ponctuation.
    tokens = re.findall(r"\bqu'|\b[cdlnmtsj]'|\w+(?:-\w+)*|[^\w\s]", texte, re.IGNORECASE)
    return tokens

def read_rules(file_name):
    """
    Lire le fichier Excel des règles de chunking et en extraire l'ensemble des règles uniques.

    Entrées :
        file_name (str) : chemin du fichier Excel contenant la colonne 'Règles'.
    Sorties :
        valeur de retour (set[str]) : ensemble des règles sans doublons.
    """
    df = read_excel(file_name)
    rules = set()

    # On élimine explicitement les doublons pour manipuler un inventaire propre des règles.
    for col in df['Règles']:
        if col not in rules:
            rules.add(col)

    return rules

def read_lexique(file_name):
    """
    Lire un lexique Excel et construire le dictionnaire mot -> catégorie grammaticale.

    Entrées :
        file_name (str) : chemin du fichier Excel contenant la colonne 'Lexique'.
    Sorties :
        valeur de retour (dict[str, str]) : dictionnaire associant chaque mot à sa catégorie.
    """
    df = read_excel(file_name)
    lexique = {}

    for col in df['Lexique']:
        # Cette entrée ne décrit pas des formes lexicales exploitables directement.
        if col == "Ensemble infini":
            continue

        # Exemple : "Prep = {sur, à, en, de}" -> catégorie = "Prep", mots = ["sur", "à", "en", "de"].
        categorie = col.split(" = ")[0]
        contenu = col.split("{")[1].split("}")[0]
        mots = [mot.strip() for mot in contenu.split(",")]

        for mot in mots:
            mot = mot.strip()
            # Si une entrée vide apparaît, elle correspond ici à la virgule retirée par le split.
            if mot == '':
                mot = ','
            lexique[mot] = categorie

    return lexique

def chercher_categorie(token, token_suivant, lexique):
    """
    Associer à un token sa catégorie.

    Entrées :
        token (str) : token courant.
        token_suivant (str | None) : token suivant (utile pour reconnaître les locutions).
        lexique (dict[str, str]) : lexique de référence.
    Sorties :
        valeur de retour (tuple[str | None, int]) : catégorie trouvée et nombre de tokens consommés.
    """
    # Cas particuliers pour les guillemets typographiques, traités comme catégories à part entière.
    if token == '\u201c':
        return 'GO', 1
    if token == '\u201d':
        return 'GF', 1

    # Tester d'abord les locutions sur deux tokens avant de retomber sur le token isolé.
    if token_suivant:
        locution = token.lower() + " " + token_suivant.lower()
        cat = lexique.get(locution, None)
        if cat:
            return cat, 2

    # Recherche directe dans le lexique.
    cat = lexique.get(token.lower(), None)
    if cat:
        return cat, 1

    # Catégorisation morphologique pour certains mots pleins.
    cat = deviner_categorie(token)
    if cat:
        return cat, 1

    return None, 1

def deviner_categorie(token):
    """
    Deviner la catégorie d'un mot plein à partir d'indices morphologiques simples.

    Entrées :
        token (str) : mot à analyser.
    Sorties :
        valeur de retour (str | None) : catégorie supposée, ou None si aucun regle ne s'applique.
    """
    mot = token.lower()
    # Les adverbes en -ment sont rattachés à la catégorie Adv.
    if re.match(r".*ment$", mot):
        return "Adv"
    # Les formes inversées comme "peut-on" ou "fait-il" sont interprétées ici comme des sujets pronominaux.
    if re.match(r".+-(on|il|elle|ils|elles)$", mot):
        return "pPersSuj"
    return None

def parse_rules(rules_set):
    """
    Transformer les règles textuelles en dictionnaire exploitable par le moteur.

    Entrées :
        rules_set (set[str]) : ensemble de règles au format source, par exemple "Det -> [N".
    Sorties :
        valeur de retour (dict[str, tuple[str, bool]]) : association déclencheur -> (type de chunk, force).
    """
    # Exemple attendu : "Det -> [N" devient {"Det": ("N", False)}.
    rules_dict = {}
    for r in rules_set:
        parties = r.split(" \u2192 [")
        declencheur = parties[0]
        type_chunk = parties[1]
        # Le marqueur "+ _" signifie que le token suivant doit être forcé dans le chunk.
        force = "+ _" in declencheur
        declencheur = declencheur.replace(" + _", "")
        rules_dict[declencheur] = (type_chunk, force)
    return rules_dict

def trouver_regle(categorie, rules_dict):
    """
    Retrouver la règle applicable à une catégorie grammaticale.

    Entrées :
        categorie (str | None) : catégorie du token courant.
        rules_dict (dict[str, tuple[str, bool]]) : dictionnaire des règles préanalysées.
    Sorties :
        valeur de retour (tuple[str | None, bool]) : type de chunk et indication de forçage du token suivant.
    """
    if categorie is None:
        return None, False
    result = rules_dict.get(categorie, None)
    if result:
        return result
    return None, False

def sauvegarder(chunk_courant, cat_courante, resultat):
    """
    Ajouter le chunk courant au résultat global.

    Entrées :
        chunk_courant (list[str]) : séquence de tokens du chunk en cours.
        cat_courante (str | None) : étiquette attribuée à ce chunk.
    Sorties :
        aucune ; la fonction modifie la liste globale chunks_resultat.
    """
    resultat.append({
        "categorie": cat_courante,
        "tokens": chunk_courant
    })

def chunker(tokens, lexique, rules_dict, resultat):
    """
    Parcourir les tokens du texte et construire les chunks selon les catégories et les règles.

    Entrées :
        tokens (list[str]) : tokens du texte.
        lexique (dict[str, str]) : lexique de catégorisation.
        rules_dict (dict[str, tuple[str, bool]]) : règles de chunking préanalysées.
    Sorties :
        aucune ; les chunks sont accumulés dans la variable globale chunks_resultat.
    """
    chunk_courant = []
    cat_courante = None
    i = 0

    while i < len(tokens):
        token = tokens[i]
        token_suivant = tokens[i+1] if i+1 < len(tokens) else None

        # Identifier la catégorie du token courant puis la règle de chunk.
        categorie, nb_tokens = chercher_categorie(token, token_suivant, lexique)
        type_chunk, force = trouver_regle(categorie, rules_dict)

        if type_chunk:
            # On ne reouvre pas de chunk nominal si le token courant doit rester dans le même groupe.
            if cat_courante == "N" and type_chunk == "N":
                chunk_courant.append(token)
                if force and i + nb_tokens < len(tokens):
                    chunk_courant.append(tokens[i + nb_tokens])
                    nb_tokens += 1
            else:
                # Dès qu'une nouvelle règle s'applique, on clôt éventuellement le chunk précédent.
                if chunk_courant:
                    sauvegarder(chunk_courant, cat_courante, resultat)
                if nb_tokens == 2:
                    chunk_courant = [token, token_suivant]
                else:
                    chunk_courant = [token]
                cat_courante = type_chunk

                # Certaines règles imposent d'inclure aussi le token suivant.
                if force and i + nb_tokens < len(tokens):
                    chunk_courant.append(tokens[i + nb_tokens])
                    nb_tokens += 1
        else:
            # Si aucun nouveau chunk n'est ouvert, on décide soit de fermer la ponctuation,
            # soit de prolonger le chunk courant.
            if cat_courante in ("Pct", "PctFin", "PctFin", "GO", "GF"):
                sauvegarder(chunk_courant, cat_courante, resultat)
                chunk_courant = [token]
                cat_courante = None
            else:
                chunk_courant.append(token)

        # L'indice avance d'un ou deux tokens selon qu'une locution a été reconnue ou non.
        i += nb_tokens

    # Le dernier chunk construit doit être conservé en sortie.
    if chunk_courant:
        sauvegarder(chunk_courant, cat_courante, resultat)

def fusionner_none(chunks):
    """
    Fusionner les chunks non étiquetés avec le chunk qui suit pour limiter les segments seuls (en un mot).

    Entrées :
        chunks (list[dict[str, object]]) : liste des chunks produits par le moteur.
    Sorties :
        valeur de retour (list[dict[str, object]]) : nouvelle liste de chunks après fusion.
    """
    resultat = []
    i = 0
    while i < len(chunks):
        if chunks[i]["categorie"] is None and i + 1 < len(chunks):
            # Le segment non classé est rattaché au chunk suivant.
            chunks[i+1]["tokens"] = chunks[i]["tokens"] + chunks[i+1]["tokens"]
            i += 1
        else:
            resultat.append(chunks[i])
            i += 1
    return resultat

def generer_xml(chunks, fichier_sortie):
    """
    Exporter les chunks dans un fichier XML.

    Entrées :
        chunks (list[dict[str, object]]) : liste des chunks.
        fichier_sortie (str) : nom du fichier XML produit.
    Sorties :
        aucune ; la fonction écrit le résultat sur ordinateur.
    """
    root = ET.Element("texte_chunke")
    for chunk in chunks:
        cat = chunk["categorie"] if chunk["categorie"] else "inconnu"
        c = ET.SubElement(root, "chunk", cat=cat)
        c.text = " ".join(chunk["tokens"])
    tree = ET.ElementTree(root)
    ET.indent(tree)
    tree.write(fichier_sortie, encoding="utf-8", xml_declaration=True)

def generer_html(chunks, fichier_sortie):
    """
    Exporter les chunks dans une page HTML simple.

    Entrées :
        chunks (list[dict[str, object]]) : liste des chunks à afficher.
        fichier_sortie (str) : nom du fichier HTML produit.
    Sorties :
        aucune ; la fonction écrit le résultat sur ordinateur.
    """
    html = """<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <style>
        body { font-family: monospace; font-size: 14px; padding: 20px; }
        .chunk { margin: 2px 0; }
        .cat { font-weight: bold; color: #1b2687; }
    </style>
</head>
<body>
"""
    for chunk in chunks:
        cat = chunk["categorie"] if chunk["categorie"] else "inconnu"
        texte = " ".join(chunk["tokens"])
        html += f'<div class="chunk">[<span class="cat">{cat}</span> {texte}]</div>\n'
    html += "</body>\n</html>"

    with open(fichier_sortie, "w", encoding="utf-8") as f:
        f.write(html)

# Étape 1 : texte initial utilisé pour construire la système de règles.
file = "article1.txt"
with open(file, mode='r', encoding='utf-8') as f:
    content = f.read()
tokens = tokeniser(content)
rules = read_rules('bd.xlsx')
rules_dict = parse_rules(rules)
lexique = read_lexique('lex.xlsx')
chunks_resultat1 = []

# Lancer le chunking.
chunker(tokens, lexique, rules_dict, chunks_resultat1)
chunks_resultat1 = fusionner_none(chunks_resultat1)
generer_xml(chunks_resultat1, "sortie1.xml")
generer_html(chunks_resultat1, "sortie1.html")

# Afficher le résultat dans le notebook.
for chunk in chunks_resultat1:
    print(f"[{chunk['categorie']}] {' '.join(chunk['tokens'])}")


[ConjSub] Alors que
[SV] vous êtes parvenus
[PN] à prendre
[N] tous les produits inscrits
[PN] sur votre liste
[PN] de courses
[PN] en seulement 23 minutes
[Pct] ,
[SV] il ne
[SV] vous reste plus
[ConjSub] qu'
[PN] à payer
[Pct] ,
[ConjCoor] et
[SV] c' est là
[ConjSub] que
[SV] vous risquez
[PN] de perdre
[N] votre avance
[Pct] ,
[ConjSub] comme
[SV] nous
[N] l' explique un chercheur
[PN] à l' origine
[PN] de cette étude
[Pct] :
[GO] “
[ConjSub] Quelle que soit
[N] votre stratégie
[Pct] ,
[N] votre intuition
[Pct] ,
[N] votre connaissance
[PN] du magasin
[ConjCoor] et
[PN] de ses employés
[Pct] ,
[SV] c' est inéluctable
[Pct] ,
[SV] vous choisirez
[ConjSub] quoi qu'
[SV] il arrive
[Pct] ,
[N] la file
[PN] d' attente
[N] la plus lente
[GF] ”
[PctFin] .
[V] Quelles sont donc
[N] les solutions
[PN] à ce problème
[PctFin] .
[ConjCoor] Et bien
[PN] selon les résultats
[PN] de cette étude
[Pct] ,
[SV] il n' y
[PN] en a
[ConjSub] qu'
[N] une
[Pct] :
[GO] “
[ConjSub] Comme
[N] les autres caiss

In [3]:

# Étape 2.1 : même langue, même source et même lexique que le premier expérimentation.
file = "article2.txt"
with open(file, mode='r', encoding='utf-8') as f:
    content = f.read()
tokens = tokeniser(content)
rules = read_rules('bd.xlsx')
rules_dict = parse_rules(rules)
lexique = read_lexique('lex.xlsx')
chunks_resultat21 = []

# Relancer le même moteur pour mesurer son comportement sur un nouveau texte.
chunker(tokens, lexique, rules_dict, chunks_resultat21)
chunks_resultat21 = fusionner_none(chunks_resultat21)
generer_xml(chunks_resultat21, "sortie2.1.xml")
generer_html(chunks_resultat21, "sortie2.1.html")

# Affichage des chunks obtenus avec le lexique initial.
for chunk in chunks_resultat21:
    print(f"[{chunk['categorie']}] {' '.join(chunk['tokens'])}")

# Étape 2.2 : même texte, mais avec un lexique enrichi.
file = "article2.txt"
with open(file, mode='r', encoding='utf-8') as f:
    content = f.read()
tokens = tokeniser(content)
rules = read_rules('bd.xlsx')
rules_dict = parse_rules(rules)
lexique = read_lexique('lex2.xlsx')
chunks_resultat22 = []

# On observe l'effet du changement de lexique à règles constantes.
chunker(tokens, lexique, rules_dict, chunks_resultat22)
chunks_resultat22 = fusionner_none(chunks_resultat22)
generer_xml(chunks_resultat22, "sortie2.2.xml")
generer_html(chunks_resultat22, "sortie2.2.html")

# Affichage des chunks obtenus avec le second lexique.
for chunk in chunks_resultat22:
    print(f"[{chunk['categorie']}] {' '.join(chunk['tokens'])}")


[GO] “
[PN] Quand j' arrive au bureau
[Pct] ,
[SV] c' est
[ConjSub] comme
[N] une libération
[GF] ”
[N] témoigne le trentenaire
[Adv] confortablement allongé aux toilettes où
[SV] il a
[N] l' habitude
[PN] de faire des microsiestes
[PctFin] .
[GO] “
[ConjSub] Je sais que je vais enfin avoir droit
[PN] à 30 bonnes minutes
[PN] de sommeil réparateur puis
[PN] à trois quarts
[PN] d' heure durant lesquels je vais somnoler
[PN] en réunion
[GF] ”
[SV] explique-t-il
[PN] en ajustant sa veste
[SV] qui lui sert
[PN] de couette
[ConjCoor] et son attaché-case
[SV] qui lui sert
[PN] de coussin
[PctFin] .
[ConjSub] Comme beaucoup
[PN] de Français
[Pct] ,
[PN] Lilian a décidé de maximiser
[N] le confort
[ConjSub] que pouvait lui apporter son lieu
[PN] de travail
[PN] en installant
[N] un hamac
[PN] dans son open space ainsi
[ConjSub] qu'
[N] un sac
[PN] de couchage
[PN] à côté
[PN] de la photocopieuse
[PN] du troisième étage où
[SV] il s' endort
[Adv] régulièrement bercé
[PN] par le bruit
[PN] de la

In [4]:

def chercher_categorie_en(token, token_suivant, lexique):
    """
    Associer une catégorie à un token anglais en réutilisant la même stratégie que pour le français.

    Entrées :
        token (str) : token courant.
        token_suivant (str | None) : token suivant (utile pour détecter les locutions).
        lexique (dict[str, str]) : lexique anglais transposé depuis le système initial.
    Sorties :
        valeur de retour (tuple[str | None, int]) : catégorie trouvée et nombre de tokens consommés.
    """
    # Les guillemets typographiques sont traités explicitement comme dans la version française.
    if token == '\u201c':
        return 'GO', 1
    if token == '\u201d':
        return 'GF', 1

    # On teste les locutions sur deux tokens.
    if token_suivant:
        locution = token.lower() + " " + token_suivant.lower()
        cat = lexique.get(locution, None)
        if cat:
            return cat, 2

    # Sinon, on cherche une entrée simple dans le lexique.
    cat = lexique.get(token.lower(), None)
    if cat:
        return cat, 1

    # Finalement, on applique une traite morphologique adaptée à l'anglais.
    cat = deviner_categorie_en(token)
    if cat:
        return cat, 1

    return None, 1

def deviner_categorie_en(token):
    """
    Deviner une catégorie grammaticale anglaise à partir d'un indice morphologique simple.

    Entrées :
        token (str) : mot à analyser.
    Sorties :
        valeur de retour (str | None) : catégorie supposée, ou None si rien n'est détecté.
    """
    mot = token.lower()
    # En anglais, les adverbes en -ly sont rattachés à la catégorie Adv.
    if re.match(r".*ly$", mot):
        return "Adv"
    return None

def chunker_en(tokens, lexique, rules_dict, resultat):
    """
    Segmenter un texte anglais en chunks en conservant les mêmes règles structurelles.

    Entrées :
        tokens (list[str]) : tokens du texte anglais.
        lexique (dict[str, str]) : lexique anglais.
        rules_dict (dict[str, tuple[str, bool]]) : règles de chunking réutilisées.
    Sorties :
        aucune ; les chunks sont accumulés dans la variable globale chunks_resultat.
    """
    chunk_courant = []
    cat_courante = None
    i = 0

    while i < len(tokens):
        token = tokens[i]
        token_suivant = tokens[i+1] if i+1 < len(tokens) else None

        # La logique de segmentation reste la même : seule l'étape d'attribution de catégorie change.
        categorie, nb_tokens = chercher_categorie_en(token, token_suivant, lexique)
        type_chunk, force = trouver_regle(categorie, rules_dict)

        if type_chunk:
            if cat_courante == "N" and type_chunk == "N":
                chunk_courant.append(token)
                if force and i + nb_tokens < len(tokens):
                    chunk_courant.append(tokens[i + nb_tokens])
                    nb_tokens += 1
            else:
                if chunk_courant:
                    sauvegarder(chunk_courant, cat_courante, resultat)
                if nb_tokens == 2:
                    chunk_courant = [token, token_suivant]
                else:
                    chunk_courant = [token]
                cat_courante = type_chunk

                if force and i + nb_tokens < len(tokens):
                    chunk_courant.append(tokens[i + nb_tokens])
                    nb_tokens += 1
        else:
            if cat_courante in ("Pct", "PctFin", "PctFin", "GO", "GF"):
                sauvegarder(chunk_courant, cat_courante, resultat)
                chunk_courant = [token]
                cat_courante = None
            else:
                chunk_courant.append(token)

        i += nb_tokens

    if chunk_courant:
        sauvegarder(chunk_courant, cat_courante, resultat)

# Étape 3.1 : texte du même type dans une autre langue, avec transposition du lexique initial.
file = "article3.txt"
with open(file, mode='r', encoding='utf-8') as f:
    content = f.read()
tokens = tokeniser(content)
rules = read_rules('bd.xlsx')
rules_dict = parse_rules(rules)
lexique = read_lexique('lex_en.xlsx')
chunks_resultat31 = []

# Le moteur conserve les mêmes règles, seul le lexique est adapté à l'anglais.
chunker_en(tokens, lexique, rules_dict, chunks_resultat31)
chunks_resultat31 = fusionner_none(chunks_resultat31)
generer_xml(chunks_resultat31, "sortie3.1.xml")
generer_html(chunks_resultat31, "sortie3.1.html")

# Affichage des chunks obtenus avec le premier lexique anglais.
for chunk in chunks_resultat31:
    print(f"[{chunk['categorie']}] {' '.join(chunk['tokens'])}")

# Étape 3.2 : même texte anglais avec une seconde version du lexique transposé.
file = "article3.txt"
with open(file, mode='r', encoding='utf-8') as f:
    content = f.read()
tokens = tokeniser(content)
rules = read_rules('bd.xlsx')
rules_dict = parse_rules(rules)
lexique = read_lexique('lex_en2.xlsx')
chunks_resultat32 = []

# On compare l'impact d'un autre lexique sur la même structure de règles.
chunker_en(tokens, lexique, rules_dict, chunks_resultat32)
chunks_resultat32 = fusionner_none(chunks_resultat32)
generer_xml(chunks_resultat32, "sortie3.2.xml")
generer_html(chunks_resultat32, "sortie3.2.html")

# Affichage des chunks obtenus avec le second lexique anglais.
for chunk in chunks_resultat32:
    print(f"[{chunk['categorie']}] {' '.join(chunk['tokens'])}")

[N] Saying the expectations
[SV] he had set
[PN] for himself
[V] were
[Adv] completely unrealistic
[Pct] ,
[PN] friends of local man James Chao expressed skepticism
[N] this morning
[PN] after the 25-year-old announced plans
[PN] to get two different things done today
[PctFin] .
[GO] “
[ConjSub] When I heard James say
[SV] he
[V] was going
[PN] to pick up some groceries
[Pct] ,
[ConjSub] that
[V] was
[SV] one thing
[Pct] ,
[ConjSub] but when
[SV] he told me
[SV] he also wanted
[PN] to do his laundry
[Pct] ,
[SV] I realized he
[V] was pushing himself way too hard
[Pct] ,
[GF] ”
[Pct] said roommate Aaron Steiner ,
[ConjSub] adding that Chao
[V] was
[Adv] apparently oblivious
[PN] to how much sustained focus
[ConjCoor] and effort would be required
[PN] to complete not just
[SV] one
[Pct] ,
[PN] but two tasks in a single 24-hour period
[PctFin] .
[GO] “
[PN] At a certain point
[Pct] ,
[SV] you ' re
[Adv] only setting yourself up
[PN] for failure
[PctFin] .
[ConjSub] If
[SV] you get
[SV] on

In [5]:
# Outil utilisé pour compter la répartition des catégories dans les sorties.
from collections import Counter

def evaluer(chunks_auto, chunks_manuel):
    """
    Comparer les chunks produits automatiquement à une référence annotée manuellement.

    Entrées :
        chunks_auto (list[dict[str, object]]) : chunks produits par le moteur.
        chunks_manuel (list[dict[str, object]]) : chunks de référence.
    Sorties :
        aucune ; la fonction affiche les mesures d'évaluation et la distribution des catégories.
    """
    nb_auto = len(chunks_auto)
    nb_manuel = len(chunks_manuel)
    print(f"Chunks attendus (référence) : {nb_manuel}")
    print(f"Chunks trouvés (automatique) : {nb_auto}")
    
    # Comparer les textes des chunks en minuscules pour éviter les différences liées à la casse.
    frontieres_auto = [" ".join(c["tokens"]).lower() for c in chunks_auto]
    frontieres_manuel = [" ".join(c["tokens"]).lower() for c in chunks_manuel]
    
    # Associer à chaque chunk sa catégorie et son texte pour l'évaluation catégorielle.
    cats_auto = [(c["categorie"], " ".join(c["tokens"]).lower()) for c in chunks_auto]
    cats_manuel = [(c["categorie"], " ".join(c["tokens"]).lower()) for c in chunks_manuel]
    
    # Frontières : compter les correspondances exactes entre chunks automatiques et manuels.
    vp_front = 0
    manuel_used = list(frontieres_manuel)
    for f in frontieres_auto:
        if f in manuel_used:
            vp_front += 1
            manuel_used.remove(f)  # Éviter de compter deux fois le même chunk de référence.
    fp_front = nb_auto - vp_front # chunks que le chunker a produits mais qui n'existent pas dans la référence
    fn_front = nb_manuel - vp_front # chunks de la référence que le chunker n'a pas trouvés
    
    # Catégories : compter les correspondances quand le texte du chunk et son étiquette coïncident.
    vp_cat = 0
    manuel_used_cat = list(cats_manuel)
    for c in cats_auto:
        if c in manuel_used_cat:
            vp_cat += 1
            manuel_used_cat.remove(c)
    fp_cat = nb_auto - vp_cat
    fn_cat = nb_manuel - vp_cat
    
    def metrics(vp, fp, fn):
        """
        Calculer la précision, le rappel et la F-mesure à partir des comptes VP, FP et FN.

        Entrées :
            vp (int) : nombre de vrais positifs.
            fp (int) : nombre de faux positifs.
            fn (int) : nombre de faux négatifs.
        Sorties :
            valeur de retour (tuple[float, float, float]) : précision, rappel et F-mesure.
        """
        precision = vp / (vp + fp) if (vp + fp) > 0 else 0
        rappel = vp / (vp + fn) if (vp + fn) > 0 else 0
        f_mesure = 2 * precision * rappel / (precision + rappel) if (precision + rappel) > 0 else 0
        return precision, rappel, f_mesure
    
    # Calculer les métriques séparément pour les frontières de chunks et pour leurs catégories.
    p_f, r_f, f_f = metrics(vp_front, fp_front, fn_front)
    p_c, r_c, f_c = metrics(vp_cat, fp_cat, fn_cat)
    
    print(f"\nFrontières")
    print(f"VP = {vp_front}, FP = {fp_front}, FN = {fn_front}")
    print(f"Précision = {p_f:.4f}")
    print(f"Rappel = {r_f:.4f}")
    print(f"F-mesure = {f_f:.4f}")
    print(f"\nCatégories")
    print(f"VP = {vp_cat}, FP = {fp_cat}, FN = {fn_cat}")
    print(f"Précision = {p_c:.4f}")
    print(f"Rappel = {r_c:.4f}")
    print(f"F-mesure = {f_c:.4f}")

    def compter_categories(chunks):
        """
        Compter le nombre de chunks par catégorie.

        Entrées :
            chunks (list[dict[str, object]]) : liste des chunks à analyser.
        Sorties :
            valeur de retour (Counter) : distribution des catégories observées.
        """
        cats = [c["categorie"] for c in chunks]
        return Counter(cats)

    # Afficher la répartition finale des catégories dans la référence et dans la sortie automatique.
    print()
    print("Référence")
    print(compter_categories(chunks_manuel))
    print("\nAutomatique")
    print(compter_categories(chunks_auto))


In [6]:
def lire_chunks_manuel(file_name):
    """
    Lire un fichier d'annotation manuelle et reconstruire la liste des chunks de référence.

    Entrées :
        file_name (str) : nom du fichier Excel contenant les colonnes 'CHK' et 'CAT CHK'.
    Sorties :
        valeur de retour (list[dict[str, object]]) : liste des chunks manuels tokenisés.
    """
    df = read_excel(file_name)
    chunks = []
    for i, row in df.iterrows():
        texte = row['CHK']
        cat = row['CAT CHK']
        # Ignorer les lignes incomplètes qui ne permettent pas de reconstruire un chunk valide.
        if texte is None or cat is None:
            continue
        texte = str(texte).strip()
        cat = str(cat).strip()
        # Normaliser les apostrophes et les guillemets comme dans la tokenisation automatique.
        texte = texte.replace("\u2019", "'").replace("\u2018", "'")
        texte = texte.replace("\u201c", chr(34)).replace("\u201d", chr(34))
        tokens = tokeniser(texte)
        chunks.append({"categorie": cat, "tokens": tokens})
    return chunks

# Évaluation du premier texte par comparaison avec l'annotation manuelle de référence.
print("Évaluation texte 1")
chunks_manuel = lire_chunks_manuel('chunks1.xlsx')
evaluer(chunks_resultat1, chunks_manuel)
print()

# Évaluation du deuxième texte avec le lexique initial.
print("Évaluation texte 2.1")
chunks_manuel = lire_chunks_manuel('chunks2.xlsx')
evaluer(chunks_resultat21, chunks_manuel)
print()

# Évaluation du deuxième texte avec le lexique enrichi.
print("Évaluation texte 2.2")
chunks_manuel = lire_chunks_manuel('chunks2.xlsx')
evaluer(chunks_resultat22, chunks_manuel)
print()

# Évaluation du troisième texte en langue transposée avec le premier lexique anglais.
print("Évaluation texte 3.1")
chunks_manuel = lire_chunks_manuel('chunks3.xlsx')
evaluer(chunks_resultat31, chunks_manuel)
print()

# Évaluation du troisième texte avec la seconde version du lexique anglais.
print("Évaluation texte 3.2")
chunks_manuel = lire_chunks_manuel('chunks3.xlsx')
evaluer(chunks_resultat32, chunks_manuel)

Évaluation texte 1
Chunks attendus (référence) : 194
Chunks trouvés (automatique) : 182

Frontières
VP = 143, FP = 39, FN = 51
Précision = 0.7857
Rappel = 0.7371
F-mesure = 0.7606

Catégories
VP = 139, FP = 43, FN = 55
Précision = 0.7637
Rappel = 0.7165
F-mesure = 0.7394

Référence
Counter({'PN': 39, 'SV': 26, 'Pct': 26, 'N': 24, 'ConjSub': 22, 'Adv': 13, 'ConjCoor': 10, 'PctFin': 10, 'V': 9, 'Adj': 7, 'GO': 4, 'GF': 4})

Automatique
Counter({'PN': 44, 'SV': 27, 'N': 26, 'Pct': 26, 'ConjSub': 23, 'PctFin': 11, 'ConjCoor': 10, 'GO': 4, 'GF': 4, 'Adv': 4, 'V': 3})

Évaluation texte 2.1
Chunks attendus (référence) : 117
Chunks trouvés (automatique) : 72

Frontières
VP = 41, FP = 31, FN = 76
Précision = 0.5694
Rappel = 0.3504
F-mesure = 0.4339

Catégories
VP = 41, FP = 31, FN = 76
Précision = 0.5694
Rappel = 0.3504
F-mesure = 0.4339

Référence
Counter({'PN': 32, 'SV': 19, 'N': 17, 'V': 9, 'ConjSub': 7, 'Adv': 7, 'Pct': 5, 'PctFin': 5, 'ConjCoor': 5, 'GO': 3, 'GF': 3, 'Adj': 3, 'NP': 2})

A